[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_4_exploration_epsilon_greedy_frozenlake.ipynb)

<div style="text-align:center">
    <h1>Exploration vs. Exploitation: the &epsilon;-greedy Strategy</h1>
</div>
<br>
<div style="text-align:center">
    <p>Every learning agent faces one dilemma before it faces any other: should it <b>exploit</b> the best action it currently knows, or <b>explore</b> a different action that might turn out to be better? This notebook builds the &epsilon;-greedy policy from first principles on Frozen Lake, shows why a fixed exploration rate is a trap, and demonstrates that <b>decaying &epsilon; over time</b> is what actually lets SARSA solve the slippery lake.</p>
</div>

### What you will build

1. The **relationship between current and future values** &mdash; the temporal-difference (TD) update that SARSA uses to pass information backwards through time.
2. The **exploration vs. exploitation** trade-off, made concrete.
3. An **&epsilon;-greedy policy** as a small, reusable function.
4. **Epsilon decay over time** &mdash; a schedule that starts curious and ends confident.
5. A controlled experiment: **fixed &epsilon; vs. decaying &epsilon;**, compared on success rate and learning curves.

Everything runs on `FrozenLake-v1` so you can watch the ideas act on a grid you can see.

In [ ]:
# Setup: install Gymnasium and fetch the shared course helpers.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('utils_frozenlake.py'):
    !pip install -qq gymnasium==1.3.0 pygame seaborn
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils_frozenlake.py

from utils_frozenlake import (plot_values, plot_policy, plot_action_values,
                              plot_stats, test_agent, evaluate_policy, seed_everything)

## Import the necessary software libraries

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Frozen Lake in one minute

Frozen Lake is a 4&times;4 grid. The agent starts at `S` (top-left), must reach the goal `G` (bottom-right), and must avoid the holes `H`. Reaching the goal gives reward **+1**; everything else gives **0**. An episode ends when the agent reaches the goal or falls in a hole.

There are **16 states** (one per cell) and **4 actions** (left, down, right, up). That small size is deliberate: it is large enough for exploration to matter, and small enough that we can print the whole value table and *see* what the agent learned.

The environment has two modes:

- `is_slippery=False` &mdash; **firm ice.** The action you choose is the action you take. Easy.
- `is_slippery=True` &mdash; **slippery ice.** With probability 2/3 you slip to a perpendicular cell instead. This is where naive exploration strategies fall apart.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
print("States :", env.observation_space.n)
print("Actions:", env.action_space.n, "  (0=Left, 1=Down, 2=Right, 3=Up)")

ACTIONS = ["Left", "Down", "Right", "Up"]

## 1. The relationship between current and future values

Reinforcement learning is the art of connecting **what happens now** to **what happens later**. The agent stores an *action-value* table $Q(s, a)$: an estimate of the total reward it expects if it takes action $a$ in state $s$ and then keeps following its policy.

The key idea &mdash; the one every TD algorithm is built on &mdash; is that the value of *this* state-action pair can be written in terms of the value of the *next* one:

$$Q(s, a) \;=\; \underbrace{r}_{\text{reward now}} \;+\; \gamma \underbrace{Q(s', a')}_{\text{value of what comes next}}$$

The discount factor $\gamma \in [0, 1)$ says how much we care about the future: $\gamma$ near 1 means a reward ten steps away is almost as good as one right now.

We never know the true $Q$, so we **nudge** our estimate toward this relationship a little at a time. Define the **TD target** and the **TD error**:

$$\text{target} = r + \gamma\, Q(s', a'), \qquad \delta = \text{target} - Q(s, a)$$

and update:

$$Q(s, a) \;\leftarrow\; Q(s, a) + \alpha\, \delta$$

where $\alpha$ is the learning rate (step size). This single line is **SARSA** &mdash; named for the five things it uses: **S**tate, **A**ction, **R**eward, next **S**tate, next **A**ction. Notice the update needs the *next action* $a'$ &mdash; and how we pick $a'$ is exactly the exploration question this notebook is about.

In [ ]:
def td_update(Q, s, a, r, s_next, a_next, done, alpha, gamma):
    """One SARSA step: nudge Q(s,a) toward  r + gamma * Q(s_next, a_next).

    When the episode has ended (done=True) there is no future, so the
    future term is dropped by multiplying it by (not done).
    """
    target = r + gamma * Q[s_next, a_next] * (not done)
    td_error = target - Q[s, a]
    Q[s, a] += alpha * td_error
    return td_error

## 2. Exploration vs. exploitation

Here is the trap. Suppose early in training the agent stumbles once into the goal by going `Right, Right, Down, Down`. Those state-action pairs now have slightly positive $Q$. If the agent **only ever exploits** (always takes the highest-$Q$ action), it will repeat that one path forever &mdash; and never discover a shorter or safer route, or on slippery ice, never learn that its "good" path mostly leads into a hole.

- **Exploitation** = take the action with the highest current estimate. Uses what you know. Earns reward now.
- **Exploration** = take some other action. Ignores what you know. Buys *information* that may earn more reward later.

An agent that never explores gets stuck with its first lucky guess. An agent that always explores never actually *uses* what it learned. We need a knob that blends the two &mdash; and, ideally, a knob that *turns* as learning progresses.

## 3. The &epsilon;-greedy policy

The simplest useful blend: flip a biased coin. With probability $\varepsilon$ **explore** (pick a uniformly random action); otherwise **exploit** (pick the greedy action).

$$
a =
\begin{cases}
\text{random action} & \text{with probability } \varepsilon \\
\arg\max_{a} Q(s, a) & \text{with probability } 1 - \varepsilon
\end{cases}
$$

- $\varepsilon = 1.0$ &rarr; pure exploration (a random walk).
- $\varepsilon = 0.0$ &rarr; pure exploitation (fully greedy).
- $\varepsilon = 0.1$ &rarr; greedy 90% of the time, curious 10% of the time.

In [ ]:
def epsilon_greedy(Q, state, epsilon, n_actions):
    """Return an action: explore with prob. epsilon, else exploit the greedy action."""
    if np.random.random() < epsilon:
        return np.random.randint(n_actions)      # EXPLORE: random action
    return int(np.argmax(Q[state]))              # EXPLOIT: best known action


# Sanity check: with epsilon=0 we always get the greedy action; with epsilon=1 it varies.
Q_demo = np.array([[0.1, 0.9, 0.2, 0.0]])   # a single state; action 1 is clearly best
greedy_choices  = [epsilon_greedy(Q_demo, 0, 0.0, 4) for _ in range(1000)]
random_choices  = [epsilon_greedy(Q_demo, 0, 1.0, 4) for _ in range(1000)]
print("epsilon=0.0  -> unique actions chosen:", set(greedy_choices), "(always the best)")
print("epsilon=1.0  -> action counts:", np.bincount(random_choices), "(roughly uniform)")

## 4. Epsilon decay over time

A fixed $\varepsilon$ forces a single compromise for the entire run. But the *right* amount of exploration changes as the agent learns:

- **Early**, the $Q$ table is all zeros &mdash; "greedy" is meaningless, so we should explore a lot.
- **Late**, the estimates are good &mdash; exploring now just throws away reward, so we should mostly exploit.

The fix is to **start with high $\varepsilon$ and decay it toward a small floor** as episodes go by. This is *simulated-annealing* logic: curious first, confident later.

We will use a simple linear decay from `epsilon_start` down to `epsilon_end`:

$$\varepsilon(\text{episode}) = \max\!\left(\varepsilon_{\text{end}},\; \varepsilon_{\text{start}}\left(1 - \frac{\text{episode}}{N}\right)\right)$$

In [ ]:
def epsilon_schedule(episode, n_episodes, epsilon_start=1.0, epsilon_end=0.01):
    """Linearly anneal epsilon from epsilon_start to a floor of epsilon_end."""
    frac = 1.0 - episode / n_episodes
    return max(epsilon_end, epsilon_start * frac)


# Visualise the schedule we will train with.
N = 20000
eps_curve = [epsilon_schedule(e, N) for e in range(N)]
plt.figure(figsize=(8, 3))
plt.plot(eps_curve)
plt.title("Epsilon decay schedule")
plt.xlabel("Episode"); plt.ylabel("epsilon (exploration rate)")
plt.axhline(0.01, ls="--", c="grey", lw=1, label="floor = 0.01")
plt.legend(); plt.tight_layout(); plt.show()

## 5. SARSA with a configurable exploration rate

We now assemble the pieces: the `epsilon_greedy` policy chooses actions, the `td_update` passes value backwards, and the `epsilon_schedule` controls curiosity over time. The `decay` flag lets us switch between the two strategies we want to compare **without changing anything else** &mdash; a clean controlled experiment.

In [ ]:
def sarsa(env, n_episodes, alpha=0.1, gamma=0.99,
          epsilon_start=1.0, epsilon_end=0.01, decay=True, seed=42):
    """Train a tabular SARSA agent with an epsilon-greedy behaviour policy.

    decay=True  -> epsilon anneals from epsilon_start to epsilon_end.
    decay=False -> epsilon is held fixed at epsilon_start for the whole run.

    Returns the learned Q table and a history dict for plotting.
    """
    seed_everything(env, seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    Q = np.zeros((n_states, n_actions))
    history = {"return": [], "epsilon": []}

    for episode in range(n_episodes):
        if decay:
            epsilon = epsilon_schedule(episode, n_episodes, epsilon_start, epsilon_end)
        else:
            epsilon = epsilon_start

        state, _ = env.reset()
        action = epsilon_greedy(Q, state, epsilon, n_actions)
        done, episode_return = False, 0.0

        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            # choose the NEXT action with the SAME epsilon-greedy policy (this is what makes it SARSA)
            next_action = epsilon_greedy(Q, next_state, epsilon, n_actions)
            td_update(Q, state, action, reward, next_state, next_action, done, alpha, gamma)
            state, action = next_state, next_action
            episode_return += reward

        history["return"].append(episode_return)
        history["epsilon"].append(epsilon)

    return Q, history


# Helpers to turn a learned Q table into a greedy policy and to score it.
def greedy_policy(Q):
    """Return a function state -> best action, for evaluation and rendering."""
    return lambda state: int(np.argmax(Q[state]))

def success_rate(env, Q, episodes=2000):
    """Average reward of the greedy policy = fraction of episodes that reach the goal."""
    return evaluate_policy(env, greedy_policy(Q), episodes)

## 6. Warm-up on firm ice

First the easy case, `is_slippery=False`. This makes the exploration point in the cleanest possible setting. We train two agents that are identical except for the exploration knob:

- **Decaying &epsilon;**: starts at 1.0 (explore everything), decays to 0.01.
- **Fixed &epsilon; = 0.1**: a "reasonable-looking" small constant that never changes.

In [ ]:
env_firm = gym.make("FrozenLake-v1", is_slippery=False)

Q_decay_firm, hist_decay_firm = sarsa(env_firm, n_episodes=6000, alpha=0.5,
                                      epsilon_start=1.0, epsilon_end=0.01, decay=True)
Q_fixed_firm, hist_fixed_firm = sarsa(env_firm, n_episodes=6000, alpha=0.5,
                                      epsilon_start=0.1, decay=False)

print("Firm ice, greedy success rate after training")
print(f"  decaying epsilon : {success_rate(env_firm, Q_decay_firm):.3f}")
print(f"  fixed epsilon=0.1: {success_rate(env_firm, Q_fixed_firm):.3f}")

On firm ice the decaying agent should reach the goal essentially every time, while the fixed-&epsilon; = 0.1 agent often gets stuck: 10% exploration from the very first episode, applied on top of an all-zeros table, wastes early steps and never builds the strong early exploration needed to first *find* the goal. Same algorithm, same learning rate &mdash; the only difference is how curiosity was scheduled.

## 7. The real test: slippery ice

Now the hard case, `is_slippery=True`, where the agent slips two-thirds of the time. Here we run a three-way comparison that reveals the **Goldilocks** nature of exploration. All three agents use the *same* SARSA code, the same 20,000-episode budget, and the same learning rate &mdash; only the exploration knob differs:

- **Too little** (`fixed &epsilon; = 0.01`): almost always greedy. On an all-zeros start it rarely wanders far enough to *find* the goal, so it never gets a learning signal.
- **Too much** (`fixed &epsilon; = 0.5`): explores plenty, but because SARSA is *on-policy*, half-random behaviour forever drags its value estimates toward a timid policy &mdash; and it never commits to the route it found.
- **Just right** (`decaying &epsilon;`, 1.0 &rarr; 0.01): explores hard early to map the lake, then anneals toward exploitation to lock in the payoff.

In [ ]:
env_slip = gym.make("FrozenLake-v1", is_slippery=True)

# Just right: explore early, exploit late.
Q_decay, hist_decay = sarsa(env_slip, n_episodes=20000, alpha=0.1,
                            epsilon_start=1.0, epsilon_end=0.01, decay=True)
# Too little exploration: nearly always greedy from the start.
Q_low,   hist_low   = sarsa(env_slip, n_episodes=20000, alpha=0.1,
                            epsilon_start=0.01, decay=False)
# Too much exploration: half-random forever.
Q_high,  hist_high  = sarsa(env_slip, n_episodes=20000, alpha=0.1,
                            epsilon_start=0.5, decay=False)

sr_decay = success_rate(env_slip, Q_decay, episodes=3000)
sr_low   = success_rate(env_slip, Q_low,   episodes=3000)
sr_high  = success_rate(env_slip, Q_high,  episodes=3000)
print("Slippery ice, greedy success rate after training")
print(f"  decaying epsilon (1.0 -> 0.01) : {sr_decay:.3f}   <- just right")
print(f"  fixed epsilon = 0.01           : {sr_low:.3f}   <- too little exploration")
print(f"  fixed epsilon = 0.50           : {sr_high:.3f}   <- too much exploration")

## 8. Compare performance: fixed &epsilon; vs decaying &epsilon;

Numbers first, then pictures. The left panel shows the smoothed learning curves; the right panel shows the final greedy success rate. The decaying agent should climb and *stay* high. The two fixed agents fail in opposite ways: `&epsilon;=0.01` flatlines near zero (it never explored enough to find the goal), while `&epsilon;=0.5` learns *something* but plateaus well below decay (a permanent random-action tax it can never switch off).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

def smooth(x, k=200):
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k)/k, mode="valid")

ax[0].plot(smooth(hist_decay["return"]), label="decaying (1.0->0.01)", c="tab:green")
ax[0].plot(smooth(hist_high["return"]),  label="fixed epsilon=0.5",    c="tab:orange")
ax[0].plot(smooth(hist_low["return"]),   label="fixed epsilon=0.01",   c="tab:red")
ax[0].set_title("Learning curve (smoothed episode return)")
ax[0].set_xlabel("Episode"); ax[0].set_ylabel("Success rate (moving avg)")
ax[0].legend()

labels = ["decay", "fixed=0.01", "fixed=0.5"]
vals   = [sr_decay, sr_low, sr_high]
colors = ["tab:green", "tab:red", "tab:orange"]
ax[1].bar(labels, vals, color=colors)
ax[1].set_title("Final greedy success rate (3000 eval episodes)")
ax[1].set_ylim(0, 1)
for i, v in enumerate(vals):
    ax[1].text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")

plt.tight_layout(); plt.show()

### Why the two failure modes?

Both fixed agents lose to decay, for **opposite** reasons &mdash; this is the whole point:

- **Too little exploration (`&epsilon;=0.01`).** Starting from an all-zeros table, an almost-greedy agent keeps repeating the same early action and rarely reaches the goal. No goal, no reward, no learning signal &mdash; it flatlines.
- **Too much exploration (`&epsilon;=0.5`).** It finds the goal easily, but SARSA learns the value of the policy it *actually follows*, and that policy is half-random forever. Its estimates stay timid and its greedy read-out is mediocre &mdash; it never commits.
- **Decaying &epsilon;.** High &epsilon; early buys the information (find the goal, map the holes); low &epsilon; late lets it exploit that information. Same algorithm, right schedule.

The practical rule that falls out of this: **explore early, exploit late.**

## 9. Look inside the learned agent

Because Frozen Lake is tiny, we can print exactly what the successful (decaying) agent learned. `plot_values` shows the state-value $V(s) = \max_a Q(s,a)$ &mdash; a heat map of "how good is it to stand here". `plot_policy` shows the greedy action in each cell as an arrow. You should see values rising toward the goal and arrows that route *around* the holes.

In [ ]:
V_decay = Q_decay.max(axis=1)
plot_values(V_decay, env=env_slip, title="V(s) learned by the decaying-epsilon agent")
plot_policy(Q_decay, env=env_slip, action_meanings={0:'L', 1:'D', 2:'R', 3:'U'})

## 10. Watch it act (optional, needs rendering)

Finally, roll out the greedy policy and watch it. This cell needs `render_mode="rgb_array"` (works in Colab). On slippery ice the agent will still occasionally slip into a hole &mdash; that is the environment's randomness, not a bug &mdash; but it should reach the goal the large majority of the time.

In [ ]:
env_render = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")
test_agent(env_render, greedy_policy(Q_decay), episodes=5)

## Summary

- **Current &harr; future values.** SARSA nudges $Q(s,a)$ toward $r + \gamma\,Q(s',a')$ using the TD error $\delta = \text{target} - Q(s,a)$. This is how reward information flows backwards through a trajectory.
- **Exploration vs. exploitation.** Pure exploitation freezes the agent on its first lucky path; pure exploration never uses what it learned. You need both.
- **&epsilon;-greedy** is the simplest blend: explore with probability $\varepsilon$, exploit otherwise.
- **Epsilon decay** matters more than the algorithm here. Start high (explore, map the world), decay to a small floor (exploit what you learned).
- **Fixed vs. decaying.** With the *same* SARSA code, decaying &epsilon; solved slippery Frozen Lake near-optimally while a fixed &epsilon; = 0.1 barely learned &mdash; because a permanent random-action tax keeps knocking a trained agent off a narrow safe path.

**Try it yourself:** change `epsilon_end`, the number of episodes, or the decay shape (e.g. exponential instead of linear) and watch the success rate move. These are exactly the knobs you would expose as sliders in an interactive teaching demo.

## Resources

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2nd ed.), Ch. 2 (bandits / &epsilon;-greedy) and Ch. 6 (TD & SARSA).
- Gymnasium Frozen Lake docs: https://gymnasium.farama.org/environments/toy_text/frozen_lake/